In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt; plt.style.use('default')
import pandas as pd
from collections import defaultdict
import pickle

data_dir = Path('./qwen_gpqa')

datasets = {
    'biology': {
        'scores': np.load(data_dir / 'biology_scores.npy')[0],
        'targets': np.load(data_dir / 'biology_targets.npy'),
    },
    'chemistry': {
        'scores': np.load(data_dir / 'chemistry_scores.npy')[0],
        'targets': np.load(data_dir / 'chemistry_targets.npy'),
    },
    'physics': {
        'scores': np.load(data_dir / 'physics_scores.npy')[0],
        'targets': np.load(data_dir / 'physics_targets.npy'),
    },
}

In [ ]:
datasets['biology']['scores'].shape

In [ ]:
alpha = 0.05  # Desired error rate
num_trials = 100  # Number of random trials to perform

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import math

def calculate_r_value(datasets, alpha, num_trials, num_rephrasings):
    all_r_value_sets = defaultdict(list)
    all_test_targets = defaultdict(list)
    all_sizes_rvalue = defaultdict(list)
    all_average_sizes = defaultdict(list)
    all_correct_predictions = defaultdict(list)

    for i in tqdm(range(num_trials)):
        for name, dataset in datasets.items():
            probabilities = dataset['scores']
            targets = dataset['targets']

            # Shuffle data
            index = np.arange(len(probabilities))
            np.random.shuffle(index)

            probabilities = probabilities[index]
            targets = targets[index]

            # Split data into validation and test sets
            validation_probabilities = probabilities[:len(probabilities) // 2]
            validation_targets = targets[:len(targets) // 2]
            test_probabilities = probabilities[len(probabilities) // 2:]
            test_targets = targets[len(targets) // 2:]

            validation_correct_probs = np.array([validation_probabilities[i, :, t] for i, t in enumerate(validation_targets)])
            val_true_probs = validation_correct_probs.T

            r_value_sets = []
            for test_sample in test_probabilities:
                all_probs = np.concatenate([val_true_probs, test_sample], axis=1)
                sorted_indices = np.argsort(-all_probs, axis=1)

                num_rows, num_cols = sorted_indices.shape
                index_counts = np.zeros(num_cols, dtype=int)
                is_used = np.zeros(num_cols, dtype=bool)

                results = []
                for col in range(num_cols):
                    np.add.at(index_counts, sorted_indices[:, col], 1)
                    masked_counts = np.where(is_used, -1, index_counts)
                    max_idx = np.argmax(masked_counts)
                    r_value = (col + 1) / num_cols
                    results.append((max_idx, r_value))
                    is_used[max_idx] = True

                results_df = pd.DataFrame(results, columns=['index', 'r_value'])
                test = results_df[results_df['index'] >= len(validation_correct_probs)]
                val = results_df[results_df['index'] < len(validation_correct_probs)]

                # print(f"Index: {round((1-alpha)*len(validation_correct_probs))}")
                # print(f"Length: {len(val)}")
                # print(f"Index 2: {min(math.ceil((len(validation_correct_probs) + 1) * (1 - alpha)), len(val)-1)}")
                # boundary = val.iloc[round((1-alpha)*len(validation_correct_probs))]['r_value']
                boundary = val.iloc[min(math.ceil((len(validation_correct_probs) + 1) * (1 - alpha)), len(val)-1)]['r_value']
                test_filtered = test[test['r_value'] <= boundary]
                test_filtered.loc[:, 'index'] = test_filtered['index'] - len(validation_correct_probs)
                r_value_sets.append({
                    'alpha': alpha,
                    'data': {key: list(value.values()) for key, value in test_filtered.to_dict().items()}
                })

            all_average_sizes[name].append(np.mean([len(r['data']['index']) for r in r_value_sets]))
            
            # Collect results
            correct_predictions = sum(1 for i, r in enumerate(r_value_sets)
                                    if test_targets[i] in r['data']['index']) / len(test_targets)

            all_r_value_sets[name].append(r_value_sets)
            all_test_targets[name].append(test_targets)
            all_sizes_rvalue[name].append([len(r['data']['index']) for r in r_value_sets])
            all_correct_predictions[name].append(correct_predictions)
    
    return all_average_sizes, all_correct_predictions

# # Report results
# print(f'R-VALUE COVERAGE at alpha: {alpha}')
# print()
# for name, results in all_correct_predictions.items():
#     print(name.center(50, '-'))
#     print(f'Coverage: {np.mean(results):.2%} +/- {np.std(results):.2%}')
#     print()

# print('********************')
# print(f'SET SIZES at alpha: {alpha}')
# print()
# for name, results in all_average_sizes.items():
#     print(name.center(50, '-'))
#     print(f'Set Size: {np.mean(results):.2f} +/- {np.std(results):.1f}')
#     print()


In [ ]:
from functools import partial
import numpy as np
import torch


def calibrate_lac(scores, targets, alpha=0.1, return_dist=False):
    """
    Estimates the 1-alpha quantile on held-out calibration data.
    The score function is `1 - max(softmax_score)`.
    
    Arguments:
        scores: softmax scores of the calibration set
        targets: corresponding labels of the calibration set
        alpha: parameter for the desired coverage level (1-alpha)

    Returns:
       qhat: the estimated quantile
       score_dist: the score distribution
    """
    scores = torch.tensor(scores, dtype=torch.float)
    targets = torch.tensor(targets)
    assert scores.size(0) == targets.size(0)
    assert targets.size(0)
    n = torch.tensor(targets.size(0))
    assert n

    score_dist = torch.take_along_dim(1 - scores, targets.unsqueeze(1), 1).flatten()
    assert (
        0 <= torch.ceil((n + 1) * (1 - alpha)) / n <= 1
    ), f"{alpha=} {n=} {torch.ceil((n+1)*(1-alpha))/n=}"
    qhat = torch.quantile(
        
        score_dist, torch.ceil((n + 1) * (1 - alpha)) / n, interpolation="higher"
    )
    return (qhat, score_dist) if return_dist else qhat


def inference_lac(scores, qhat, allow_empty_sets=False):
    """
    Makes prediction sets on new test data
    
    Arguments:
        scores: softmax scores of the test set
        qhat: estimated quantile of the calibration set from the `calirbate_lac` function
        allow_empty_sets: if True allow a prediction set to contain no predictions (will then satisfy upper bound of marginal coverage)

    Returns:
       prediction_sets: boolean mask of prediction sets (True if class is included in the prediction set; otherwise False)
    """
    scores = torch.tensor(scores, dtype=torch.float)
    n = scores.size(0)

    elements_mask = scores >= (1 - qhat)

    if not allow_empty_sets:
        elements_mask[torch.arange(n), scores.argmax(1)] = True
        
    prediction_sets = elements_mask

    return prediction_sets



def get_coverage(psets, targets, precision=None):
    """
    Calculates empirical coverage of prediction sets
    
    Arguments:
        psets: prediction sets of test set
        targets: ground true labels of test set
        precision: rounding precision

    Returns:
       coverage: how many times the answer is in the prediction set
    """
    psets = torch.tensor(psets)
    targets = torch.tensor(targets)
    psets = psets.clone()
    targets = targets.clone()
    n = psets.shape[0]
    coverage = psets[torch.arange(n), targets].float().mean().item()
    # if precision is not None:
    #     coverage = round(coverage, precision)
    return coverage


def get_size(psets, precision=1):
    """
    Calculates empirical set sizes of prediction sets (can consider as the average uncertainty of the model)
    
    Arguments:
        psets: prediction sets of test set
        precision: rounding precision

    Returns:
       size: how many prediction does each set contain on average
    """
    psets = psets.clone()
    size = psets.sum(1).float().mean().item()
    # if precision is not None:
    #     size = round(size, precision)
    return size


from collections import defaultdict

def get_lac(datasets, alpha, num_trials):
    all_q_hats = defaultdict(list)
    all_scores = defaultdict(list)
    all_psets = defaultdict(list)
    all_targets = defaultdict(list)
    all_coverage = defaultdict(list)
    all_size = defaultdict(list)

    for i in range(num_trials):
        for name, results in datasets.items():
            scores = results['scores'][:, 0, :]  
            targets = results['targets']
            
            # shuffle data
            index = np.arange(len(scores))
            np.random.shuffle(index)

            # split data into calibration and test sets
            
            n = len(scores) // 2
            # n = int(0.05 * len(scores))
            cal_scores = scores[index][:n]
            cal_targets = targets[index][:n]
            val_scores = scores[index][n:]
            val_targets = targets[index][n:]

            # estimate 1-alpha quantile on calibration set
            q = calibrate_lac(cal_scores, cal_targets, alpha=alpha)

            # make prediction sets on test set
            psets = inference_lac(val_scores, q)
            
            all_psets[name].append(psets)
            all_scores[name].append(val_scores)
            all_targets[name].append(val_targets)
            
            # Calculate coverage and set size for this trial
            coverage = get_coverage(psets, val_targets)
            size = get_size(psets)
            
            all_coverage[name].append(coverage)
            all_size[name].append(size)
            
    return all_coverage, all_size


def get_lac_using_mean(datasets, alpha, num_trials):
    all_q_hats = defaultdict(list)
    all_scores = defaultdict(list)
    all_psets = defaultdict(list)
    all_targets = defaultdict(list)
    all_coverage = defaultdict(list)
    all_size = defaultdict(list)

    for i in range(num_trials):
        for name, results in datasets.items():
            scores = results['scores'].mean(axis=1)
            targets = results['targets']
            
            # shuffle data
            index = np.arange(len(scores))
            np.random.shuffle(index)

            # split data into calibration and test sets
            
            n = len(scores) // 2
            # n = int(0.05 * len(scores))
            cal_scores = scores[index][:n]
            cal_targets = targets[index][:n]
            val_scores = scores[index][n:]
            val_targets = targets[index][n:]

            # estimate 1-alpha quantile on calibration set
            q = calibrate_lac(cal_scores, cal_targets, alpha=alpha)

            # make prediction sets on test set
            psets = inference_lac(val_scores, q)
            
            all_psets[name].append(psets)
            all_scores[name].append(val_scores)
            all_targets[name].append(val_targets)
            
            # Calculate coverage and set size for this trial
            coverage = get_coverage(psets, val_targets)
            size = get_size(psets)
            
            all_coverage[name].append(coverage)
            all_size[name].append(size)
            
    return all_coverage, all_size

# print(f'COVERAGE at alpha: {alpha}')
# print()

# for name in datasets.keys():
#     print(name.center(50, '-'))
#     mean_coverage = np.mean(all_coverage[name])
#     std_coverage = np.std(all_coverage[name])
#     print(f'{mean_coverage:.2%} +/- {std_coverage:.0%}')
#     print()

# print('********************')
# print(f'SET SIZES at alpha: {alpha}')
# print()

# for name in datasets.keys():
#     print(name.center(50, '-'))
#     mean_size = np.mean(all_size[name])
#     std_size = np.std(all_size[name])
#     print(f'{mean_size:.2f} +/- {std_size:.1f}')
#     print()

In [ ]:
models = ['llama1b_gpqa', 'llama3b_gpqa', 'mistral_gpqa', 'phi_gpqa', 'qwen_gpqa']
all_results = []

for model in models:
    data_dir = Path(f'./{model}')

    datasets = {
        'biology': {
            'scores': np.load(data_dir / 'biology_scores.npy')[0],
            'targets': np.load(data_dir / 'biology_targets.npy'),
        },
        'chemistry': {
            'scores': np.load(data_dir / 'chemistry_scores.npy')[0],
            'targets': np.load(data_dir / 'chemistry_targets.npy'),
        },
        'physics': {
            'scores': np.load(data_dir / 'physics_scores.npy')[0],
            'targets': np.load(data_dir / 'physics_targets.npy'),
        },
    }
    alpha = 0.05  # Desired error rate
    num_trials = 100  # Number of random trials to perform   
    
    all_sizes_rvalue, all_correct_predictions = calculate_r_value(datasets, alpha, num_trials, num_rephrasings=21)
    all_coverage, all_size = get_lac(datasets, alpha, num_trials)
    all_coverage_using_mean, all_size_using_mean = get_lac_using_mean(datasets, alpha, num_trials)

    lac_coverage = defaultdict(list)
    for name in datasets.keys():
        lac_coverage[name] = np.mean(all_coverage[name])
    r_value_coverage = defaultdict(list)
    for name in datasets.keys():
        r_value_coverage[name] = np.mean(all_correct_predictions[name])
    lac_set_sizes = defaultdict(list)
    for name in datasets.keys():
        lac_set_sizes[name] = np.mean(all_size[name])
    r_value_set_sizes = defaultdict(list)
    for name in datasets.keys():
        r_value_set_sizes[name] = np.mean(all_sizes_rvalue[name])
    lac_using_mean_coverage = defaultdict(list)
    for name in datasets.keys():
        lac_using_mean_coverage[name] = np.mean(all_coverage_using_mean[name])
    lac_using_mean_set_sizes = defaultdict(list)
    for name in datasets.keys():
        lac_using_mean_set_sizes[name] = np.mean(all_size_using_mean[name])
    


    all_results.append({
        'lac_coverage': lac_coverage,
        'r_value_coverage': r_value_coverage,
        'lac_set_sizes': lac_set_sizes,
        'r_value_set_sizes': r_value_set_sizes,
        'lac_using_mean_coverage': lac_using_mean_coverage,
        'lac_using_mean_set_sizes': lac_using_mean_set_sizes,
    })
    

In [ ]:
import pandas as pd
from pathlib import Path

# Create a directory to save CSV files
csv_results_dir = Path('./csv_results_gpqa')
csv_results_dir.mkdir(exist_ok=True)

# Iterate over the models and their corresponding results
for model, result in zip(models, all_results):
    # Create a DataFrame for coverage
    coverage_df = pd.DataFrame({
        'regular': result['lac_coverage'],
        'mean': result['lac_using_mean_coverage'],
        'R': result['r_value_coverage'],
    })
    
    # Create a DataFrame for set sizes
    set_size_df = pd.DataFrame({
        'regular': result['lac_set_sizes'],
        'mean': result['lac_using_mean_set_sizes'],
        'R': result['r_value_set_sizes'],
    })
    
    # Save the coverage DataFrame to CSV
    coverage_csv_path = csv_results_dir / f'{model}_coverage_alpha_005_gpqa.csv'
    coverage_df.to_csv(coverage_csv_path, index=True)
    
    # Save the set sizes DataFrame to CSV
    set_size_csv_path = csv_results_dir / f'{model}_set_size_alpha_005_gpqa.csv'
    set_size_df.to_csv(set_size_csv_path, index=True)

    print(f"Saved CSV files for {model}:")
    print(f" - Coverage: {coverage_csv_path}")
    print(f" - Set sizes: {set_size_csv_path}")

In [ ]:
# take the mean of the results across trials keep the dictionary structure
lac_coverage = defaultdict(list)
for name in datasets.keys():
    lac_coverage[name] = np.mean(all_correct_predictions[name])
r_value_coverage = defaultdict(list)
for name in datasets.keys():
    r_value_coverage[name] = np.mean(all_coverage[name])
lac_set_sizes = defaultdict(list)
for name in datasets.keys():
    lac_set_sizes[name] = np.mean(all_size[name])
r_value_set_sizes = defaultdict(list)
for name in datasets.keys():
    r_value_set_sizes[name] = np.mean(all_average_sizes[name])
    

# lac_coverage = [np.mean(all_coverage[name]) for name in datasets.keys()]
# r_value_coverage = [np.mean(all_correct_predictions[name]) for name in datasets.keys()]

# lac_set_sizes = [np.mean(all_size[name]) for name in datasets.keys()]
# r_value_set_sizes = [np.mean(all_average_sizes[name]) for name in datasets.keys()]

# save the results
model = 'qwen_gpqa'
alpha = alpha
results_dir = Path('./results')
results_dir.mkdir(exist_ok=True)
with open(results_dir / f'{model}_alpha_{alpha}.pkl', 'wb') as f:
    pickle.dump({
        'lac_coverage': lac_coverage,
        'r_value_coverage': r_value_coverage,
        'lac_set_sizes': lac_set_sizes,
        'r_value_set_sizes': r_value_set_sizes,
    }, f)


In [ ]:
r_value_set_sizes